# 02 · Retrieval failure modes

An eval table is a summary; the failures are where the system reveals what it's actually doing. This notebook walks through five kinds of failures we observed during evaluation, with concrete query-and-result examples for each.

If you're a hiring manager skimming: this is the bit that separates "I tuned a model" from "I understand the model." Treat the failures as load-bearing — they explain what the system *can't* do, which is more diagnostic than what it can.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, str(Path.cwd().parent / "src"))

from poi.embeddings.factory import build_encoder
from poi.index import FaissIndex
from poi.retrieval import RetrievalPipeline
from poi.utils.config import POIConfig

INDEX_PATH = Path("../artifacts/celeba_default.index")
CONFIG_PATH = Path("../configs/default.yaml")

In [ ]:
cfg = POIConfig.from_yaml(CONFIG_PATH)
cfg.vlm.enabled = False  # we're inspecting raw retrieval, no captions

encoder = build_encoder(cfg.embedding)
index = FaissIndex.load(INDEX_PATH)
pipeline = RetrievalPipeline(encoder=encoder, index=index, cfg=cfg.retrieval)

In [ ]:
def show_results(query: str, top_k: int = 8) -> None:
    """Display the top-k matches for a query as a horizontal strip."""
    response = pipeline.search(query, top_k=top_k)
    fig, axes = plt.subplots(1, top_k, figsize=(top_k * 2, 2.5))
    for ax, hit in zip(axes, response.hits):
        ax.imshow(Image.open(hit.image_path))
        ax.set_title(f"#{hit.rank}  {hit.score:.2f}", fontsize=9)
        ax.axis("off")
    plt.suptitle(query, fontsize=11)
    plt.tight_layout()
    plt.show()

## Failure 1 · Negation

CLIP-family models, including SigLIP-2, do not handle "not" cleanly. The token contributes to the embedding but doesn't reliably *invert* the concept it modifies.

In [ ]:
show_results("A man with glasses but no beard.")
show_results("A man with glasses, clean-shaven, no facial hair.")  # rephrased without negation

Compare the two strips. The first one returns plenty of bearded men with glasses; the second is much cleaner. The fix is at the prompt level — replace negations with positive descriptions of the desired property. A query rewriter (small LLM, runs in <100ms) is the production answer.

## Failure 2 · Compositional binding

The encoder knows the parts but doesn't reliably bind them. "Red dress" + "wide-brimmed hat" gets you a soup of red things and hat things, not always the combination.

In [ ]:
show_results("A woman wearing a red dress and a wide-brimmed hat.")

This is one of the well-known weaknesses of bi-encoder retrieval. Multi-vector representations (ColBERT family) handle composition better because each query token can independently attend to image regions. On the next-steps list.

## Failure 3 · Numerical attributes (age)

Age estimation from a single photo is hard, and CelebA is heavily skewed young. Both contribute.

In [ ]:
show_results("A woman in her sixties with gray hair.")

The system finds gray hair fine; the age estimate is unreliable. Some of the top hits look 40, not 60. This is at least partly a corpus problem — CelebA's age distribution is concentrated in the 20–40 range.

## Failure 4 · Out-of-distribution queries

CelebA is single-person face crops. Anything that requires multi-person scenes or full-body context is out of distribution by construction.

In [ ]:
show_results("Two people standing next to each other.")
show_results("A person wearing a full suit.")

These return whatever happens to be the closest single-face crop, with low confidence scores across the board. A production system should detect this and flag it ("this query may be out of distribution for the corpus") rather than silently returning bad matches.

## Failure 5 · Stylistic / contextual descriptions

This is the most worrying failure mode because the system *appears* to work.

In [ ]:
# We don't run this query in the notebook output, deliberately.
# Stereotype-coded prompts are a fairness concern, not a feature.
# show_results("Looks like a librarian.")

When we ran prompts like "looks like a [profession]" during development, the results matched stereotypical visual coding — exactly the bias the model picked up from web-scale training data. We deliberately don't ship those prompts as examples in the UI, because the system's surface-level competence at this kind of query is a bug in the world being modeled, not a feature.

If you're building on top of this for production, an explicit denylist of profession/role/demographic prompts is cheap insurance. A more principled fix is debiasing at the embedding level, which is an open research area.

## Summary

Five failure modes, each with a different fix:

| Failure | Fix |
|---|---|
| Negation | Query rewriting with a small LLM |
| Compositional binding | Late-interaction (ColBERT-style) reranker |
| Numerical attributes | Different corpus, or task-specific head |
| Out-of-distribution | OOD detection, route to a different system |
| Stylistic/biased queries | Denylist + debiasing, ongoing care |

Three of these are next-steps in the README. Two are not — they're caveats to keep in mind any time you deploy a CLIP-family retrieval system.